# CARMA-JAX in AMBRS

[CARMA-JAX](https://github.com/reflective-org/carma-jax) is a JAX port of the CARMA
sectional aerosol model: the size distribution lives on a geometric mass grid (here 47
mass-doubling bins, radii ~0.2 nm to 8 µm), and coagulation moves particle number between
bins with a physically-derived Brownian kernel.

This adapter wires CARMA-JAX's production-quality **coagulation** path; growth and
nucleation exist on its dev branch but aren't yet exported by the installed package, so
requesting them raises rather than silently doing nothing.

Like the other JAX models it runs **in process** — no binaries, so this notebook works
straight after `pip install -r requirements.txt`.

1. define a scenario,
2. watch pure coagulation reshape the size distribution over a day,
3. check the invariants (number falls, mass exactly conserved),
4. see how bin resolution changes the answer — a knob sectional models uniquely expose,
5. notes on comparing against the other AMBRS models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import ambrs
import ambrs.aerosol as aerosol
import ambrs.gas as gas

plt.rcParams.update({
    "figure.figsize": (7.2, 4.4),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,          # recessive grid
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# a fixed, colourblind-safe order (Okabe-Ito); assigned in order, never cycled
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#E69F00"]


def size_distribution(output, diameters):
    """dN/dlnD [# m^-3] for an ambrs Output, on the given diameter grid [m]."""
    return np.asarray(output.compute_variable("dNdlnD", {
        "diam_grid": diameters,
        "normalize": False,
        "wetsize": False,
        "method": "hist",
    }))


def bin_diameters(model):
    """CARMA's own bin-centre diameters [m] — evaluate distributions on these,
    never on a finer grid than the model resolves."""
    return 2.0 * np.asarray(model.r) / 100.0   # bin radius [cm] -> D [m]


def summarise(label, output):
    population = output.particle_population
    return {
        "case": label,
        "N [# m^-3]": population.get_Ntot(),
        "dry mass [kg m^-3]": population.get_tot_dry_mass(),
    }


print("carma-jax available:", ambrs.carma_jax._CARMA_JAX_AVAILABLE)

## 1. Define a scenario

A polluted two-mode sulfate/organic aerosol. CARMA carries one internally-mixed element,
so the two modes' compositions get blended (volume-weighted) — the adapter warns when that
blend is actually lossy, i.e. when the modes differ.

In [ ]:
so4 = aerosol.AerosolSpecies(name="SO4", molar_mass=97.071, density=1770,
                             hygroscopicity=0.507)
oc = aerosol.AerosolSpecies(name="OC", molar_mass=12.01, density=1000,
                            hygroscopicity=0.1)
h2so4 = gas.GasSpecies(name="H2SO4", molar_mass=98.079)


def mode(name, number, gmd, gsd, species, fractions):
    return aerosol.AerosolModeState(
        name=name, species=tuple(species), number=number,
        geom_mean_diam=gmd, log10_geom_std_dev=np.log10(gsd),
        mass_fractions=tuple(fractions))


scenario = ambrs.Scenario(
    aerosols=(so4, oc),
    gases=(h2so4,),
    size=aerosol.AerosolModalSizeState(modes=(
        mode("aitken",       5e11, 3.0e-8, 1.6, [so4],     [1.0]),
        mode("accumulation", 1e11, 1.2e-7, 1.6, [so4, oc], [0.7, 0.3]),
    )),
    gas_concs=(0.0,),
    flux=0.0,
    relative_humidity=0.5,
    temperature=288.0,         # [K]
    pressure=101325.0,         # [Pa]
    height=500.0,              # [m]
)

model = ambrs.carma_jax.AerosolModel(ambrs.AerosolProcesses(coagulation=True))
print(f"bin grid: {model.nbin} bins, radii "
      f"{model.r[0]*1e7:.2f} nm to {float(model.r[-1])*1e4:.1f} um")

## 2. A day of pure coagulation

Sampled at 0, 1, 6 and 24 hours, plotted on CARMA's own bin-centre diameters — sampling a
sectional model finer than it resolves draws grid artifacts, not physics.

In [ ]:
DT = 60.0                      # [s]
HOURS = [0, 1, 6, 24]
inputs = [model.create_input(scenario, dt=DT, nstep=int(h * 3600 / DT) or 1)
          for h in HOURS]
# nstep must be >= 1, so "hour 0" is a single step: effectively the initial state
outputs = model.run_ensemble(inputs)

D = bin_diameters(model)
fig, ax = plt.subplots()
for colour, hours, output in zip(PALETTE, HOURS, outputs):
    ax.plot(D * 1e9, size_distribution(output, D), lw=2,
            color=colour, label=f"{hours} h")
ax.set_xscale("log")
ax.set_xlim(2, 3000)
ax.set_xlabel("dry diameter [nm]")
ax.set_ylabel("dN/dlnD [# m$^{-3}$]")
ax.set_title("A day of coagulation (CARMA-JAX)")
ax.legend(title="elapsed", frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame([summarise(f"{h} h", o) for h, o in zip(HOURS, outputs)]).set_index("case")

## 3. The invariants

Coagulation merges particles: total number must fall monotonically, and — because mass just
moves between bins — total dry mass must not change at all. The mass column above is flat to
machine precision; here is the number decay.

In [ ]:
sample_hours = np.array([0, 0.5, 1, 2, 4, 8, 16, 24])
series = [model.run(model.create_input(scenario, dt=DT,
                                       nstep=int(h * 3600 / DT) or 1))
          for h in sample_hours]
numbers = np.array([o.particle_population.get_Ntot() for o in series])
masses = np.array([o.particle_population.get_tot_dry_mass() for o in series])

fig, ax = plt.subplots()
ax.plot(sample_hours, numbers, lw=2, marker="o", ms=5, color=PALETTE[0])
ax.set_yscale("log")
ax.set_xlabel("elapsed [h]")
ax.set_ylabel("total number [# m$^{-3}$]")
ax.set_title("Number decays; mass stays put")
plt.tight_layout()
plt.show()

print("max relative mass drift over 24 h: %.2e" %
      float(np.max(np.abs(masses - masses[0]) / masses[0])))

## 4. Bin resolution — a sectional model's own knob

The same scenario on 16, 24 and 47 bins covering the *same* size span — coarser grids get
a proportionally larger mass ratio between bins, otherwise fewer bins would just mean a
smaller grid. Coarse grids smear the distribution and shift where coagulation puts the
mass; this numerical-resolution sensitivity is invisible in a modal model (whose "grid" is
fixed by construction) and is exactly the kind of structural difference AMBRS exists to
expose.

In [ ]:
# same total mass span as the 47-bin default (2**46), redistributed over fewer bins
resolutions = [(16, 2.0 ** (46 / 15)), (24, 2.0 ** (46 / 23)), (47, 2.0)]
fig, ax = plt.subplots()
rows = []
for colour, (nbin, rmrat) in zip(PALETTE, resolutions):
    res_model = ambrs.carma_jax.AerosolModel(
        ambrs.AerosolProcesses(coagulation=True), nbin=nbin, rmrat=rmrat)
    output = res_model.run(res_model.create_input(
        scenario, dt=DT, nstep=int(6 * 3600 / DT)))
    D_res = bin_diameters(res_model)
    ax.plot(D_res * 1e9, size_distribution(output, D_res), lw=2,
            color=colour, label=f"{nbin} bins")
    rows.append(summarise(f"{nbin} bins", output))
ax.set_xscale("log")
ax.set_xlim(2, 3000)
ax.set_xlabel("dry diameter [nm]")
ax.set_ylabel("dN/dlnD [# m$^{-3}$]")
ax.set_title("6 h of coagulation at three bin resolutions")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

pd.DataFrame(rows).set_index("case")

The distributions agree on the broad answer and diverge in the details: coarse grids smear
the peak and fatten the large tail. The mass column is the sharper lesson — in a one-moment
sectional scheme every particle in a bin carries the bin-centre mass, so a coarse grid
misstates *total mass* from the very first step (here by ~80% at 16 bins), even though
number is placed exactly. Two-moment schemes (like TOMAS) track mass separately precisely
to avoid this.

## 5. Comparing against the other AMBRS models

Every model returns the same `ambrs.analysis.Output`, so the plotting above works
unchanged on any of them. Two comparisons this model makes natural:

- **Sectional vs sectional:** TOMAS-JAX (two-moment, 40 mass-doubling bins) against
  CARMA-JAX (one-moment, configurable bins) under identical coagulation-only scenarios —
  the difference isolates the moment treatment and kernel formulation.
- **Sectional vs modal:** either against MAM4/MAM4-JAX, where mode merging replaces
  explicit inter-bin transfer.

```python
tomas = ambrs.tomas_jax.AerosolModel(ambrs.AerosolProcesses(coagulation=True))
carma = ambrs.carma_jax.AerosolModel(ambrs.AerosolProcesses(coagulation=True))
outs = [m.run(m.create_input(scenario, dt=60.0, nstep=1440))
        for m in (tomas, carma)]
```

`ambrs.analysis.kl_divergence(a, b)` and `nmae([a], [b], "dNdlnD")` quantify the
difference. (TOMAS-JAX and MAM4-JAX land in separate PRs; once merged, the snippet above
runs as-is.)